In [ ]:
!nvidia-smi

In [ ]:
!env | grep LD_LIBRARY_PATH

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/scratch/a5q/alexr.a5q/.cache/huggingface/hub/models--unsloth--Llama-3.2-1B-Instruct"
# resolve to the actual snapshot dir
import os

refs = os.path.join(model_path, "refs", "main")
if os.path.exists(refs):
    sha = open(refs).read().strip()
    model_path = os.path.join(model_path, "snapshots", sha)

print(f"Loading from: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, local_files_only=True, dtype=torch.float16, device_map="cuda"
)
print(f"Model loaded on {model.device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

In [ ]:
messages = [
    {"role": "user", "content": "What is the capital of France? Reply in one sentence."}
]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
).to("cuda")
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=50)
response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
)
print(response)